# Indian Legal Judgment — Domain-Specific Text Analysis & Retrieval System
### NLP Assessment 1 | Vidyashilp University — Seventh Semester 2026-27
**Framework:** Select &rarr; Order &rarr; Implement &rarr; Compare &rarr; Evaluate &rarr; Justify
**Domain:** Indian Supreme Court & High Court Case Judgments (25 Digital Case Documents, D01–D25)

---
This master notebook serves as a complete, sequential, reproducible laboratory manual covering all requirements of Assessment 1.

## 1. Project Objective and Legal-Domain Justification
The objective of this system is to analyze, process, and retrieve Indian legal judgments using tailored NLP pipelines.
Legal text exhibits distinct linguistic characteristics that break conventional NLP assumptions:
- **Statutory Citations:** e.g., *Section 302 IPC*, *Article 226*, *NDPS Act s.20* contain periods, slashes, numbers, and acronyms.
- **Critical Negation Polarity:** Words like *not*, *no*, *without*, *unless* dictate legal culpability (*not guilty* vs *guilty*).
- **Morphological Ambiguities:** Polysemous terms like *learned counsel* (honorific adjective vs past verb) or *execution* (decree execution vs executive branch).
- **Precedent Citations & Currency:** *AIR 2020 SC 123*, *₹50,000* bond.

In [1]:
# Setup environment and import core modules
import sys
from pathlib import Path
import pandas as pd

# Ensure project root is in sys.path
BASE_DIR = Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from config import (
    DOCUMENT_STATISTICS_CSV, PREPROCESSING_RESULTS_CSV, TOKENIZATION_COMPARISON_CSV,
    STEMMING_LEMMATIZATION_CSV, POS_TAGGING_RESULTS_CSV, CUSTOM_POS_RESULTS_CSV,
    NER_RESULTS_CSV, NGRAM_RESULTS_CSV, BPE_RESULTS_CSV, INVERTED_INDEX_JSON,
    RETRIEVAL_RESULTS_CSV, PIPELINE_COMPARISON_CSV, EVALUATION_RESULTS_CSV
)
from src.document_loader import DocumentLoader
from src.tokenizers import CustomLegalTokenizer, NLTKTokenizer, SpacyTokenizer, HybridTokenizer, compare_tokenizers
from src.bpe_analyzer import BPEAnalyzer
from src.stopwords_handler import StopwordsHandler
from src.stemming_lemmatization import StemmingLemmatizationAnalyzer
from src.pos_tagger import DefaultPOSTagger
from src.custom_pos_tagger import RuleBasedLegalPOSTagger, MLCustomPOSTagger, generate_custom_pos_comparison_df
from src.ner_processor import LegalNERProcessor
from src.ngram_analyzer import NGramAnalyzer
from src.pipeline_a import PipelineA
from src.pipeline_b import PipelineB
from src.inverted_index import PositionalInvertedIndex
from src.retrieval import LegalRetrievalEngine
from src.evaluation import RelevanceEvaluator, BENCHMARK_QUERIES

print('Environment and modules loaded successfully!')

Environment and modules loaded successfully!


## 2. Document Loading, Extraction & Corpus Statistics (Module 1)
Loading 25 Indian High Court and Supreme Court judgment PDFs and linking them with authoritative metadata.

In [2]:
loader = DocumentLoader()
documents = loader.load_documents()
stats_df, summary = loader.generate_statistics_df()

print(f'Total Documents Loaded: {len(documents)}')
print('Corpus Summary Metrics:')
for k, v in summary.items():
    if k != 'Counting conventions':
        print(f'  {k}: {v}')

display(stats_df[['Document ID', 'File Name', 'Case Name', 'Pages', 'Tokens', 'Vocabulary Size']].head(8))

Total Documents Loaded: 25
Corpus Summary Metrics:
  Number of documents: 25
  Number of sentences: 9402
  Number of tokens: 257849
  Number of characters: 1351861
  Vocabulary size: 10819
  Average document length: 10313.96


,Document ID,File Name,Case Name,Pages,Tokens,Vocabulary Size
0,D01,case1177.PDF,Sonu Gupta v. State of U.P.,6,2440,596
1,D02,case0683.PDF,Parmod Devgan @ Ashu v. State of Haryana,4,1165,354
2,D03,case0625.PDF,Vashist Tiwari v. State of U.P.,3,1010,314
3,D04,case0707.PDF,Prashanth Kumar M v. State of Karnataka,3,855,311
4,D05,case0647.PDF,Hu Xiaolin v. State of Karnataka,4,1487,376
5,D06,case1178.PDF,"Tarun Kumar v. Asst. Director, ED",12,6515,1236
6,D07,case0414.PDF,Deepak Kochhar v. Directorate of Enforcement,37,23823,2524
7,D08,case1181.PDF,Sanjay Agarwal v. Directorate of Enforcement,11,6315,1227


## 3. Tokenization Comparison & BPE Subwords (Module 2, Exercises 1–4)
Comparing five tokenization approaches:
1. NLTK `word_tokenize`
2. spaCy linguistic tokenizer
3. Custom Legal Tokenizer
4. Byte Pair Encoding (BPE)
5. Hybrid Tokenizer

In [3]:
bpe_analyzer = BPEAnalyzer()
corpus_texts = [d['cleaned_text'] for d in documents.values()]
bpe_analyzer.train_on_corpus(corpus_texts)

sample_legal_phrases = [
    'Section 302 IPC',
    'AIR 2020 SC 123',
    '₹50,000/- personal bond',
    'Article 21 and Article 226',
    'Crl.A. No. 123/2023',
    'suo motu criminal-appeal'
]
tok_records = compare_tokenizers(sample_legal_phrases, bpe_tokenizer=bpe_analyzer.bpe_wrapper)
tok_df = pd.DataFrame(tok_records)
display(tok_df[['Input_Text', 'Custom_Tokens', 'NLTK_Tokens', 'spaCy_Tokens', 'BPE_Tokens', 'Hybrid_Tokens']])

,Input_Text,Custom_Tokens,NLTK_Tokens,spaCy_Tokens,BPE_Tokens,Hybrid_Tokens
0,Section 302 IPC,Section 302 IPC,Section | 302 | IPC,Section | 302 | IPC,Section | 302 | IPC,Section 302 IPC
1,AIR 2020 SC 123,AIR 2020 | SC | 123,AIR | 2020 | SC | 123,AIR | 2020 | SC | 123,AIR | 2020 | SC | 12 | 3,AIR 2020 | SC | 123
2,"₹50,000/- personal bond","₹50,000/- | personal | bond","₹50,000/- | personal | bond","₹ | 50,000/- | personal | bond","[UNK] | 50 | , | 000 | /- | personal | bond","₹50,000/- | personal | bond"
3,Article 21 and Article 226,Article 21 | and | Article 226,Article | 21 | and | Article | 226,Article | 21 | and | Article | 226,Article | 21 | and | Article | 22 | 6,Article 21 | and | Article 226
4,Crl.A. No. 123/2023,Crl.A. No. 123/2023,Crl.A | . | No | . | 123/2023,Crl | . | A. | No | . | 123/2023,Crl | . | A | . | No | . | 12 | 3 | / | 2023,Crl.A. No. 123/2023
5,suo motu criminal-appeal,suo motu | criminal-appeal,suo | motu | criminal-appeal,suo | motu | criminal | - | appeal,su | o | m | ot | u | criminal | - | appeal,suo motu | criminal-appeal


### BPE Subword Formation Analysis

In [4]:
bpe_results_df = bpe_analyzer.analyze(corpus_texts)
display(bpe_results_df[['Word / Expression', 'Category', 'BPE Tokens', 'Subword Formation Behavior']].head(10))

,Word / Expression,Category,BPE Tokens,Subword Formation Behavior
0,the,Common general,the,Single unified token
1,court,Common legal,court,Single unified token
2,order,Common legal,order,Single unified token
3,appeal,Common legal,appeal,Single unified token
4,accused,Common legal,accused,Single unified token
5,jurisprudential,Rare lexical,juris | pr | ud | ential,"Segmented into 4 subword units: ['juris', 'pr'..."
6,quashment,Rare morphological,quash | ment,"Segmented into 2 subword units: ['quash', 'ment']"
7,circumstantiality,Rare morphological,circum | stan | ti | ality,"Segmented into 4 subword units: ['circum', 'st..."
8,non-bailable,Hyphenated domain,non | - | bailable,"Segmented into 3 subword units: ['non', '-', '..."
9,unconstitutionality,Rare morphological,unconstitutional | ity,Segmented into 2 subword units: ['unconstituti...


## 4. Stopwords, Stemming & Lemmatization Experiments (Module 2, Exercises 7–10)
Investigating over-stemming, under-stemming, and the legal rationale for preserving negation terms.

In [5]:
stem_lemma_analyzer = StemmingLemmatizationAnalyzer()
sl_df = stem_lemma_analyzer.compare_words()
display(sl_df[['Word', 'Contextual POS', 'Porter Stem', 'Snowball Stem', 'WordNet Lemma', 'Phenomenon / Notes']].head(10))

,Word,Contextual POS,Porter Stem,Snowball Stem,WordNet Lemma,Phenomenon / Notes
0,appellants,NOUN,appel,appel,appellant,Accurate Lemmatization (retains exact legal si...
1,convicted,VERB,convict,convict,convict,Normal
2,proceedings,NOUN,proceed,proceed,proceeding,Normal
3,execution,NOUN,execut,execut,execution,Over-stemming (confounds executive branch with...
4,executive,ADJ,execut,execut,executive,Over-stemming (confounds executive branch with...
5,prosecutrix,NOUN,prosecutrix,prosecutrix,prosecutrix,Under-stemming (fails to link with prosecute/p...
6,anticipatory,ADJ,anticipatori,anticipatori,anticipatory,Normal
7,judgments,NOUN,judgment,judgment,judgment,Normal
8,jurisdiction,NOUN,jurisdict,jurisdict,jurisdiction,Normal
9,statutory,ADJ,statutori,statutori,statutory,Normal


## 5. POS Tagging & Custom POS Taggers (Module 2, Exercise 12)
Evaluating baseline NLTK vs Rule-Based and ML-Based Custom POS taggers on legal terms.

In [6]:
custom_pos_df, ml_metrics = generate_custom_pos_comparison_df()
print('ML POS Classifier Test Set Performance:')
print(f"  Accuracy: {ml_metrics['accuracy']*100:.2f}%")
print(f"  Weighted F1: {ml_metrics['f1_score']:.4f}")
print(f"  Train/Test Tokens: {ml_metrics['train_tokens']} / {ml_metrics['test_tokens']}")

display(custom_pos_df[['Word', 'Default POS (NLTK)', 'Custom Rule POS', 'Custom ML POS', 'Expected POS', 'Correction Explanation']])

ML POS Classifier Test Set Performance:
  Accuracy: 76.67%
  Weighted F1: 0.7051
  Train/Test Tokens: 119 / 30


,Word,Default POS (NLTK),Custom Rule POS,Custom ML POS,Expected POS,Correction Explanation
0,bail,NN,NN,NN,NN,"Bail is a substantive legal right/noun, not a ..."
1,learned,VBN,JJ,JJ,JJ,Honorific judicial adjective modifying counsel...
2,quashing,NN,NN,NN,NN,Gerund nominal functioning as head of noun phrase
3,suo,NN,RB,RB,RB,Latin adverbial modifier indicating 'on its ow...
4,detenu,NN,NN,NN,NN,Specialized Indian legal noun for detained person
5,bench,NNP,NN,NNP,NN,"Judicial forum/collegiate bench noun, not furn..."
6,non-bailable,JJ,JJ,JJ,JJ,Compound statutory adjective qualifying crimin...


## 6. Named Entity Recognition (Module 2, Exercise 13)
Extracting general entities and legal domain entities (Courts, Statutes, Sections, Citations).

In [7]:
ner_proc = LegalNERProcessor()
ner_df = ner_proc.generate_evaluation_df()
display(ner_df[['Source Text Excerpt', 'Predicted Entity', 'Predicted Type', 'Expected Type', 'Correctness']].head(8))

,Source Text Excerpt,Predicted Entity,Predicted Type,Expected Type,Correctness
0,The High Court of Allahabad dismissed the crim...,High Court of Allahabad,LEGAL_COURT,LEGAL_COURT,Correct
1,The High Court of Allahabad dismissed the crim...,Sonu Gupta,PERSON,PERSON,Correct
2,The accused was charged under Section 302 of t...,Section 302,LEGAL_SECTION,LEGAL_SECTION,Correct
3,The accused was charged under Section 302 of t...,Indian Penal Code,LEGAL_STATUTE,LEGAL_STATUTE,Correct
4,The trial was conducted in New Delhi on 15 Jan...,New Delhi,GPE,GPE,Correct
5,The trial was conducted in New Delhi on 15 Jan...,15 January 2024,DATE,DATE,Correct
6,The applicant was granted bail upon furnishing...,"₹50,000",MONEY,MONEY,Correct
7,The Directorate of Enforcement initiated proce...,Directorate of Enforcement,ORG,ORG,Correct


## 7. N-Gram Analysis (Module 2, Exercise 14)
Calculating unigrams through five-grams across the full 25 judgment corpus.

In [8]:
ngram_analyzer = NGramAnalyzer()
ngram_results = ngram_analyzer.analyze_corpus(corpus_texts, max_n=5)

summary_data = []
for n in range(1, 6):
    summary_data.append({
        'N-Gram': ngram_results[n]['n_label'],
        'Total Count': ngram_results[n]['total_count'],
        'Unique Count': ngram_results[n]['unique_count'],
        'Top 3 Phrases': '; '.join([f"{' '.join(g)} ({c})" for g, c in ngram_results[n]['top_10'][:3]])
    })
display(pd.DataFrame(summary_data))

,N-Gram,Total Count,Unique Count,Top 3 Phrases
0,Unigram,234381,9543,the (20133); of (12512); to (5344)
1,Bigram,234380,69527,of the (3875); in the (1596); by the (1110)
2,Trigram,234379,133134,indian kanoon http (436); kanoon http indianka...
3,4-Gram,234378,171507,indian kanoon http indiankanoon (436); kanoon ...
4,5-Gram,234377,191067,indian kanoon http indiankanoon org (436); kan...


## 8. Dual Pipeline Implementation & Inverted Index Creation (Modules 3 & 4)
Implementing Pipeline A (Conventional Baseline) and Pipeline B (Legal Optimized).

In [9]:
# Execute Pipeline A
pipe_a = PipelineA()
corpus_a = pipe_a.process_corpus(documents)
index_a = PositionalInvertedIndex()
index_a.build_index(corpus_a)
engine_a = LegalRetrievalEngine(index_a, documents, pipe_a.normalize_query_term)

# Execute Pipeline B
pipe_b = PipelineB()
corpus_b = pipe_b.process_corpus(documents)
index_b = PositionalInvertedIndex()
index_b.build_index(corpus_b)
engine_b = LegalRetrievalEngine(index_b, documents, pipe_b.normalize_query_term)

print(f'Pipeline A Vocabulary: {len(index_a.index):,} unique indexed stems')
print(f'Pipeline B Vocabulary: {len(index_b.index):,} unique indexed lemmas')
print('Postings for "bail" in Pipeline B:', len(index_b.get_postings('bail')), 'documents')

Pipeline A Vocabulary: 8,355 unique indexed stems
Pipeline B Vocabulary: 9,000 unique indexed lemmas
Postings for "bail" in Pipeline B: 25 documents


## 9. Information Retrieval: Keyword, Phrase, Boolean & Ranked Search
Testing search capabilities on Indian legal queries with snippet extraction.

In [10]:
test_queries = [
    ('bail', 'keyword'),
    ('"anticipatory bail"', 'phrase'),
    ('bail AND appeal', 'boolean'),
    ('bail AND NOT murder', 'boolean'),
    ('commercial quantity charas', 'ranked')
]

for q_text, q_type in test_queries:
    res = engine_b.search(q_text, query_type=q_type)
    print(f"\nQuery: '{q_text}' [{q_type.upper()}] -> {res['num_results']} hits in {res['execution_time_ms']} ms")
    for hit in res['results'][:2]:
        print(f"   [{hit['document_id']}] {hit['case_name']} (Score: {hit['score']})")
        print(f"   Snippet: {hit['snippet'][:120]}...")


Query: 'bail' [KEYWORD] -> 25 hits in 0.23 ms
   [D01] Sonu Gupta v. State of U.P. (Score: 1.0)
   Snippet: ...Ashok Kumar Bench: Ashok Kumar HIGH COURT OF JUDICATURE AT ALLAHABAD ?In Chamber Case :- CRIMINAL MISC. BAIL APPLICAT...
   [D02] Parmod Devgan @ Ashu v. State of Haryana (Score: 1.0)
   Snippet: ...for the complainant. *** AVNEESH JHINGAN , J.(Oral) 1. This fourth petition is filed seeking regular bail in case of ...

Query: '"anticipatory bail"' [PHRASE] -> 6 hits in 0.14 ms
   [D07] Deepak Kochhar v. Directorate of Enforcement (Score: 1.0)
   Snippet: ...16-BA.1322.2020 IN THE HIGH COURT OF JUDICATURE AT BOMBAY CRIMINAL APPELLATE JURISDICTION CRIMINAL BAIL APPLICATION N...
   [D08] Sanjay Agarwal v. Directorate of Enforcement (Score: 1.0)
   Snippet: ...SLP No.6940 of 2019 which was also dismissed on 11th March, 2019. The petitioner was also enlarged on bail by the lea...

Query: 'bail AND appeal' [BOOLEAN] -> 16 hits in 0.11 ms
   [D06] Tarun Kumar v. Asst. Director, ED (

## 10. Relevance Evaluation & Pipeline Comparison (Modules 5 & 6)
Evaluating 14 domain-specific queries against ground truth relevance labels.

In [11]:
evaluator = RelevanceEvaluator()
df_a, summary_a = evaluator.evaluate_engine(engine_a, pipeline_label='Pipeline A (Baseline)')
df_b, summary_b = evaluator.evaluate_engine(engine_b, pipeline_label='Pipeline B (Legal Optimized)')

comp_df = pd.read_csv('results/pipeline_comparison.csv')
print('=== PIPELINE COMPARISON (PIPELINE B CONFIRMED BEST MODEL) ===')
display(comp_df)

=== PIPELINE COMPARISON (PIPELINE B CONFIRMED BEST MODEL) ===


,Measure,Pipeline A (Baseline),Pipeline B (Legal Optimized),Final Pipeline
0,Token Count,129181,137860,137860
1,Vocabulary Size,8355,9000,9000
2,Mean Precision,0.7241,0.7241,0.7241
3,Mean Recall,0.8825,0.8825,0.8825
4,Mean F1-Score,0.763,0.763,0.763
5,Mean Precision@5,0.7976,0.7976,0.7976
6,Mean Execution Time (ms),0.14,0.1,0.1
7,Meaningful N-Grams Retained,Partial (Porter truncates legal stems),High (Preserves legal citations & valid lemmas),High (Pipeline B preserves legal semantics)
8,Domain Citation Integrity,Low (Section numbers and citations fragmented),"High (Citations, Acts, and Sections preserved ...",High (Full Statutory Integrity)


## 11. Final Validation & GUI Launch Instructions
To launch the interactive Streamlit user interface locally on Windows:
```powershell
# Navigate to project folder
cd Domain_Text_Analysis_Retrieval

# Launch Streamlit application
python -m streamlit run app.py
```

All empirical result files (`results/*.csv`, `results/*.json`) and the academic report (`Report.pdf`) have been generated and validated.